# 06 — Calibration Quality Assessment and Reprojection Error

**Objective:** Deeply analyse calibration quality — identify which images hurt calibration, how many views are needed, and what reprojection error means geometrically.

## What reprojection error measures

After estimating K, D, R, t for each view, we project the **known** 3-D board points back onto the image and measure distance from **detected** 2-D corners.

- **Low RMS (< 0.5 px):** Calibration is geometrically consistent. Images are sharp, diverse, and the board is planar.
- **High RMS (> 1.0 px):** At least one of these is true: blurry images, too few diverse poses, bent checkerboard, inconsistent board size setting.


In [ ]:
import sys, logging
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt

SRC = Path('..') / 'src'
if str(SRC.resolve()) not in sys.path:
    sys.path.insert(0, str(SRC.resolve()))

from calibvision.calibrator import CameraCalibrator
from calibvision.utils import load_images_from_dir, draw_reprojection_error

logging.basicConfig(level=logging.WARNING)

PROJECT_ROOT = Path('..').resolve()
CALIB_DIR    = PROJECT_ROOT / 'data' / 'calibration_images'
BOARD_COLS, BOARD_ROWS = 9, 6

# Re-run calibration
paths, _ = load_images_from_dir(CALIB_DIR)
calibrator = CameraCalibrator(board_cols=BOARD_COLS, board_rows=BOARD_ROWS, min_valid_frames=10)
detections = calibrator.detect_corners(paths, verbose=False)
result = calibrator.calibrate(detections)

print(f'Baseline RMS: {result.rms_reprojection_error:.4f} px  ({len(result.valid_image_paths)} images)')

In [ ]:
# ── 6.1  Per-view error chart ─────────────────────────────────────────────────
names = [p.name for p in result.valid_image_paths]
fig = draw_reprojection_error(result.per_view_errors, names, result.rms_reprojection_error)
plt.show()

In [ ]:
# ── 6.2  How does RMS change as we add more images? ──────────────────────────
# This experiment shows the calibration stability vs number of views.

valid_dets = [d for d in detections if d.success]
min_imgs = 10
rms_vs_n = []

for n in range(min_imgs, len(valid_dets) + 1):
    subset = valid_dets[:n]
    try:
        r = calibrator.calibrate(subset, image_size=result.image_size)
        rms_vs_n.append((n, r.rms_reprojection_error))
    except Exception:
        pass

ns, rms_vals = zip(*rms_vs_n) if rms_vs_n else ([], [])

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(ns, rms_vals, 'o-', color='#3498db', linewidth=2, markersize=5)
ax.axhline(1.0, color='#e74c3c', linestyle='--', label='1.0 px limit')
ax.axhline(0.5, color='#2ecc71', linestyle='--', label='0.5 px target')
ax.set_xlabel('Number of calibration images')
ax.set_ylabel('RMS reprojection error (px)')
ax.set_title('Calibration stability vs. number of views')
ax.legend()
plt.tight_layout()
plt.show()

print('Observation: RMS typically stabilises after ~15 diverse images.')
print('Adding more images beyond that rarely improves accuracy significantly.')

In [ ]:
# ── 6.3  Leave-one-out analysis: which image helps / hurts most? ──────────────
baseline_rms = result.rms_reprojection_error
loo_results = []

for i, det in enumerate(valid_dets):
    subset = [d for j, d in enumerate(valid_dets) if j != i]
    if len(subset) < 10:
        continue
    try:
        r = calibrator.calibrate(subset, image_size=result.image_size)
        delta = r.rms_reprojection_error - baseline_rms
        loo_results.append((det.image_path.name, delta, r.rms_reprojection_error))
    except Exception:
        pass

loo_results.sort(key=lambda x: x[1])

print('Leave-one-out analysis — impact of removing each image:')
print(f'{"Image":<30} {"ΔRMS":>10}  {"Note"}')
print('-' * 65)
for name, delta, rms_val in loo_results:
    note = '✓ helps' if delta > 0 else '✗ hurts'
    print(f'{name:<30} {delta:>+10.4f}  {note}  (RMS without: {rms_val:.4f})')

print(f'\nBaseline RMS (all images): {baseline_rms:.4f} px')
print('A positive ΔRMS means removing this image worsens calibration (it was useful).')
print('A negative ΔRMS means removing this image improves calibration (it was harmful).')

In [ ]:
# ── 6.4  Visualise reprojected corners on a sample image ──────────────────────
# Compare detected (ground truth) vs reprojected (estimated) corners.

sample_idx = 0
det = valid_dets[sample_idx]
rvec = result.rotation_vecs[sample_idx]
tvec = result.translation_vecs[sample_idx]

objp = np.zeros((BOARD_ROWS * BOARD_COLS, 3), np.float32)
objp[:, :2] = np.mgrid[0:BOARD_COLS, 0:BOARD_ROWS].T.reshape(-1, 2) * 25.0

proj_pts, _ = cv2.projectPoints(
    objp, rvec, tvec, result.camera_matrix, result.dist_coeffs
)

img = cv2.imread(str(det.image_path))
vis = img.copy()

for detected, reprojected in zip(det.corners, proj_pts):
    d_pt = tuple(detected.ravel().astype(int))
    r_pt = tuple(reprojected.ravel().astype(int))
    cv2.circle(vis, d_pt, 5, (0, 255, 0), -1)      # green = detected
    cv2.circle(vis, r_pt, 4, (0, 100, 255), -1)     # blue = reprojected
    cv2.line(vis, d_pt, r_pt, (255, 50, 50), 1)     # red line = error vector

# Scale up for visibility
h, w = vis.shape[:2]
scale = min(900 / w, 600 / h)
vis_small = cv2.resize(vis, (int(w*scale), int(h*scale)))

fig, ax = plt.subplots(figsize=(13, 7))
ax.imshow(cv2.cvtColor(vis_small, cv2.COLOR_BGR2RGB))
ax.set_title(
    f'{det.image_path.name}\n'
    f'Green = detected · Blue = reprojected · Red = error vector  '
    f'(per-view error: {result.per_view_errors[sample_idx]:.4f} px)',
    fontsize=9
)
ax.axis('off')
plt.tight_layout()
plt.show()

## Summary

| Analysis | Completed |
|---|---|
| Per-view error chart | ✓ |
| RMS stability vs number of views | ✓ |
| Leave-one-out image ranking | ✓ |
| Reprojected vs detected corner overlay | ✓ |

## Next step

Open `07_error_analysis.ipynb` for failure case analysis and parameter sensitivity.
